In [1]:
from core.database.supabase_client import get_supabase_client

In [3]:
def push_url(urls:list[str],category:str):
    client=get_supabase_client()
    rows=[{"video_url":url,"category":category} for url in urls]
    client.table("urls_data").insert(rows).execute()
    return True
    

In [7]:
def check_if_urls_exists(urls):
  try:
    exists_urls=[]
    for url in urls:
        supabase=get_supabase_client()
        response = (
        supabase.table("urls_data")
        .select("*")
        .match({"video_url": url})
        .execute()
)
        if response.data[0]:
            exists_urls.append(url)
    return exists_urls
  except Exception as e:
    print(e)
    return []

In [28]:
links=["abc","abc"]

In [29]:
set(links)

{'abc'}

In [30]:
def search_query_push(query:list[str],category:str):
    """
    search for youtube videos for given query and then push those urs into database if they are not already there
    args:
        query: list[str]
        category: str
    returns:
         A dictionary containing the count of urls pushed.
    """
    urls=set(query_videos(query))
    check_urls=check_if_urls_exists(urls)
    unique_urls=[url for url in urls if url not in check_urls]
    push_url(unique_urls,category)
    return {"count_pushed":len(unique_urls)}

In [31]:
def query_videos(queries):
    links_total=[]
    for query in queries:
        conn = http.client.HTTPSConnection("google.serper.dev")
        payload = json.dumps({
            "q": query,
            "location": "India",
            "gl": "in",
            "num": 30
        })
        headers = {
            'X-API-KEY': 'f1a587f5e82e8e768b808ebbe49deb2d7060a3ce',
            'Content-Type': 'application/json'
        }
        conn.request("POST", "/videos", payload, headers)
        res = conn.getresponse()
        data = res.read()
        resp = json.loads(data)
        links = [v["link"] for v in resp.get("videos", []) if "link" in v]
        links_total.extend(links)
    return links_total

In [25]:
from yt_rag.llm_service.gemini_client import get_gemini_client

In [40]:
system_prompt = """
You are a Data Collection Agent for YouTube URLs.

Goal:
- For category {category}, collect up to {TARGET_PER_CATEGORY} unique YouTube video URLs and store them in Supabase table `urls_data` (video_url, category).

Tools:
- search_query_push(query: str, category: str) returns {{"count_pushed": int}} and ensures de-dup + insert.


- Generate multiple query variants (tutorial, course, guide, lecture, tips, beginner, advanced, year ranges, synonyms).
- Prefer queries targeting watch pages (site:youtube.com/watch). Do not invent URLs.
- Call search_query_push(query, category) iteratively until:
  - target reached, or
  - 3 consecutive attempts yield 0 new URLs.
Be idempotent. Let the tool handle DB deduplication.
Avoid NSFW and off-topic results.
Summarize outcome per category and overall at the end.

If a tool call fails or returns nothing, adapt the query and retry.
"""

In [32]:
from langchain_core.tools import tool

In [35]:

tools = [search_query_push]

In [34]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

In [46]:
video_categories = [
    # --------------- EDUCATIONAL FOCUS (10 types) ---------------
    "Academic Lectures & Courses",          # School/college-level subjects (Physics, Math, Biology)
    "Competitive Exam Prep",                # GATE, UPSC, JEE, NEET
    "Coding & Tech Tutorials",              # Python, ML, Data Science, Web Dev
    "Career Guidance & Skill Development",  # Resume, jobs, internships, freelancing
    "Language Learning & Communication",    # Spoken English, IELTS, grammar
    "Finance & Business Education",         # Personal finance, accounting, stock market tutorials
    "Science & Research Explainers",        # Concepts, experiments, innovations
    "Engineering & Practical Labs",         # Circuits, mechanical, civil, etc.
    "Education News & Policy Discussions",  # NEP updates, education reforms
    "Motivational & Study Productivity",    # Study techniques, time management, student motivation

    # --------------- MAJOR GENERAL CATEGORIES ---------------
    "Podcasts & Interviews",                # TRS, Raj Shamani, long-form discussions
    "News & Current Affairs",               # NDTV, WION, The Print
    "Tech Reviews & Gadgets",               # Technical Guruji, Beebom
    "Spiritual & Self-Improvement",         # Sadhguru, Gaur Gopal Das
    "Lifestyle & Vlogs",                    # Flying Beast, Sejal Kumar
    "Comedy & Entertainment",               # Zakir Khan, FilterCopy
    "Documentaries & Storytelling",         # Curly Tales, NatGeo India
    "Food & Cooking",                       # Hebbars Kitchen, Chef Ranveer Brar
    "Travel & Culture",                     # Tanya Khanijow, Nomadic Indian
    "Fitness & Health"                      # Fit Tuber, Guru Mann
]

In [47]:
import json
from collections import defaultdict
from typing import List, Dict
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain.chat_models import init_chat_model
import http

In [48]:
@tool
def search_query_push(query: List[str], category: str) -> Dict:
    """Search YouTube for given queries and push unique URLs into DB."""
    urls = list(set(query_videos(query)))         
    existing = set(check_if_urls_exists(urls))
    unique = [u for u in urls if u not in existing]
    push_url(unique, category)
    return {
        "count_pushed": len(unique),
        "pushed_urls": unique[:25] 
    }

tools = [search_query_push]

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")
llm = llm.bind_tools(tools)

In [ ]:
TARGET = 100
CATEGORY = "coding tutorials"
USER_QUERY = "Find and push coding tutorial videos popular with Indian viewers in English."

In [ ]:
messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=USER_QUERY),
]

In [56]:
def run_agent(category):
    try:
        total_pushed = 0
        max_steps = 20 
        step = 0
        TARGET = 100
        CATEGORY = category
        USER_QUERY = f"Find and push {CATEGORY} videos popular with Indian viewers in English."
        system_prompt = f"""
        You are a data sourcer. Goal: push at least {TARGET} unique, valid YouTube video URLs
        for CATEGORY="{CATEGORY}". Work in batches:
        1) Propose focused search queries (English, India audience).
        2) CALL the tool search_query_push(query, category) with 10–30 queries at a time.
        3) After the tool returns, if total < {TARGET}, refine and CALL AGAIN with *new* queries.
        4) Never fabricate URLs. Use only queries; the tool resolves URLs and dedupes in DB.
        5) Prefer general English queries targeting India (avoid Hindi terms unless explicitly asked).
        6) Stop once cumulative pushed >= {TARGET}.
        7) focus on diversification of data collection
        Return brief progress notes between calls.
        """
        messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=USER_QUERY),
        ]

        while total_pushed < TARGET and step < max_steps:
            step += 1
            ai = llm.invoke(messages)

            if getattr(ai, "tool_calls", None):
                for call in ai.tool_calls:
                    name = call["name"]
                    args = call["args"]
                    call_id = call["id"]

                    if name == "search_query_push":
                        result = search_query_push.invoke(args)  
                        pushed = int(result.get("count_pushed", 0))
                        total_pushed += pushed

                        messages.append(ai) 
                        messages.append(
                            ToolMessage(
                                name=name,
                                content=json.dumps({
                                    "step": step,
                                    "count_pushed_this_step": pushed,
                                    "total_pushed_so_far": total_pushed
                                }),
                                tool_call_id=call_id,
                            )
                        )
                continue

            messages.append(ai)
            break
        print("Total pushed: ", total_pushed)
        return True
    except Exception as e:
        print(e)
        return False
        

In [57]:
for category in video_categories:
    resp=run_agent(category)

list index out of range
Total pushed:  565
list index out of range
Total pushed:  816
list index out of range
Total pushed:  845
list index out of range
Total pushed:  854
list index out of range
Total pushed:  488
list index out of range
Total pushed:  861
list index out of range
Total pushed:  870
list index out of range
Total pushed:  860
list index out of range
Total pushed:  620
list index out of range
Total pushed:  693
list index out of range
Total pushed:  577
list index out of range
Total pushed:  526
list index out of range
Total pushed:  865
list index out of range
Total pushed:  852
list index out of range
Total pushed:  817
list index out of range
Total pushed:  635
list index out of range
Total pushed:  648
list index out of range
Total pushed:  773
list index out of range
Total pushed:  727
list index out of range
Total pushed:  855


In [58]:
import pandas as pd

In [59]:
data=pd.read_csv("urls_data_rows.csv")

In [ ]:
dupilicates=data[data['video_url'].duplicated()==False]

In [72]:
data.drop_duplicates(subset=["video_url"],inplace=True)

In [83]:
def is_yt_vid(url):
    return url.startswith("https://www.youtube.com/watch?v=")

In [75]:
test_url="https://www.youtube.com/watch?v=ZX7HavT-i3g"

In [ ]:
is_yt_vid(test_url)

True

In [85]:
filtered_data=data.where(data['video_url'].apply(is_yt_vid)).dropna()

In [90]:
filtered_data.groupby("category").count().sum()

video_url    11965
dtype: int64

In [97]:
filtered_data.to_csv("filtered_data.csv")

In [92]:
gemini=get_gemini_client()

In [96]:
links=[url for url in filtered_data['video_url']]

In [98]:
filtered_data_2=pd.read_csv("filtered_data.csv")


In [99]:
links

['https://www.youtube.com/watch?v=__yqhaqRfqw',
 'https://www.youtube.com/watch?v=_-3DI6GXOHo',
 'https://www.youtube.com/watch?v=_-4TKFInXwE',
 'https://www.youtube.com/watch?v=_-km7L2z0W4',
 'https://www.youtube.com/watch?v=_-mGUom77dQ',
 'https://www.youtube.com/watch?v=_-NWyuUAAK4',
 'https://www.youtube.com/watch?v=_-T3YYFFytI',
 'https://www.youtube.com/watch?v=_0hEPok2Jw4',
 'https://www.youtube.com/watch?v=_0mT78NUo-A',
 'https://www.youtube.com/watch?v=_19sQY5pna8',
 'https://www.youtube.com/watch?v=_1LmirXDC-E',
 'https://www.youtube.com/watch?v=_1N7jS4PmHE',
 'https://www.youtube.com/watch?v=_1pz5ZPSnrM',
 'https://www.youtube.com/watch?v=_1tMumZAETk',
 'https://www.youtube.com/watch?v=_1u9JGNn9SY',
 'https://www.youtube.com/watch?v=_2Z78RNVXXo',
 'https://www.youtube.com/watch?v=_3fstqCt--o',
 'https://www.youtube.com/watch?v=_3WgbGL7f5I',
 'https://www.youtube.com/watch?v=_4ELdcVZgYk',
 'https://www.youtube.com/watch?v=_4XxoUc9qxk',
 'https://www.youtube.com/watch?v=_5Fzau